# Browse val-set imagery per city

Renders every tile of the **val shard caches** (`val_cities_cache`, `val_temporal_cache`)
city by city, in pages, so you can eyeball what the model sees in the cities where
within-city Spearman is great (Colorado Springs 0.92, Lexington 0.90) vs terrible
(Dayton -0.07, Tulsa 0.01, Augusta 0.09) — see `src/diagnose_bracket_gap.py`.

Each tile shows its **label z**, the model's **prediction** (from the CSV dumped by
`diagnose_bracket_gap.py --preds-out`), the **residual**, year and GEOID. Tiles are
sorted by `SORT_BY` (default: label) so a functioning city should show a visible
poor-to-rich gradient across pages — if Dayton's gradient looks identical everywhere,
the imagery isn't legible there; if the imagery clearly varies but predictions don't
track it, the problem is elsewhere (labels, vintage, ...).

**Usage**: set `CITY = None` in the config cell to get the city menu + summary table,
then set `CITY` (cbsa code or a title substring like `"Dayton"`) and run the render
cell. `PAGES = None` renders *all* pages for the city; or use the optional slider cell
at the bottom (needs `pip install ipywidgets`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.diagnose_bracket_gap import load_val_shards
from src.utils.paths import PROCESSED_DATA_DIR, RESULTS_DIR

# ── Config ──────────────────────────────────────────────────────────────
CACHE_ROOT = Path(os.environ.get("CACHE_DIR", "~/data/cache")).expanduser()
CACHE_DIRS = [CACHE_ROOT / "val_cities_cache", CACHE_ROOT / "val_temporal_cache"]
PREDS_CSV = RESULTS_DIR / "run_20260710" / "val_bracket_preds.csv"  # or None: no predictions
SPLITS_FEATHER = PROCESSED_DATA_DIR / "cbsa_splits.feather"

CITY = "Dayton"        # None -> show city menu; else cbsa code (int) or title substring ("Dayton")
SORT_BY = "label"  # label | pred | residual | year
PAGE_SIZE = 24     # tiles per page (rows of 6)
PAGES = None       # None -> all pages of CITY; or e.g. [0, 1]

In [ ]:
# ── Load shards, attach predictions + city names, per-city summary ─────────────────
df, IMAGES = load_val_shards(CACHE_DIRS)
df = df.reset_index(drop=True)
df["img_idx"] = np.arange(len(df))

if PREDS_CSV is not None and Path(PREDS_CSV).exists():
    preds = pd.read_csv(PREDS_CSV)
    key = ["set", "building_id", "year"]
    preds = preds.drop_duplicates(subset=key)
    df = df.merge(preds[key + ["predicted_value"]], on=key, how="left")
    print(f"predictions merged: {df['predicted_value'].notna().sum()}/{len(df)} tiles")
else:
    df["predicted_value"] = np.nan
    print("⚠️ no predictions CSV — browsing imagery + labels only")
df["residual"] = df["predicted_value"] - df["Rel_Score"]

splits = pd.read_feather(SPLITS_FEATHER)
TITLE = {int(c): t for c, t in zip(splits["cbsa_code"], splits["cbsa_title"])}
BRACKET = {int(c): b for c, b in zip(splits["cbsa_code"], splits["bracket"])}
df["city"] = df["cbsa_code"].astype(int).map(TITLE).fillna("?")
df["bracket"] = df["cbsa_code"].astype(int).map(BRACKET).fillna("?")

def _rho(g):
    from scipy.stats import spearmanr
    ok = g.dropna(subset=["predicted_value"])
    if len(ok) < 3 or ok["Rel_Score"].nunique() < 2 or ok["predicted_value"].nunique() < 2:
        return np.nan
    return spearmanr(ok["predicted_value"], ok["Rel_Score"])[0]

# n_img counts building x year TILES (not independent!); n_bld = unique buildings
# (~tracts) is the effective cross-sectional sample. rho pools all years (same
# convention as compute_val_metrics / the wandb bracket metrics); rho_bld ranks
# per-building mean pred vs mean label — one observation per building.
def _city_summary(g):
    by_bld = g.groupby("building_id")[["predicted_value", "Rel_Score"]].mean()
    return pd.Series({
        "n_img": len(g), "n_bld": g["building_id"].nunique(),
        "rho": _rho(g),
        "rho_bld": _rho(by_bld.rename(columns={"predicted_value": "predicted_value",
                                               "Rel_Score": "Rel_Score"}))
                   if len(by_bld) >= 5 else np.nan,
        "label_std": g["Rel_Score"].std(),
    })

SUMMARY = (df.groupby(["set", "cbsa_code", "city", "bracket"])
             .apply(_city_summary, include_groups=False)
             .reset_index().sort_values("rho"))
print(f"\n{len(df)} tiles, {df['cbsa_code'].nunique()} cities. Worst rho first "
      f"(trust rho_bld/n_bld — tiles repeat buildings across years):")
display(SUMMARY.round(3))

In [ ]:
# ── Rendering helpers ───────────────────────────────────────────────────────────
# NOTE: shard "geoids" are hash surrogates (main.py CyclicCacheManager:
# hash(GEOID) % 2**31), used only to exclude same-tract pairs — NOT FIPS codes.
# Tiles therefore show building_id, which joins to
# data/processed/pair_buildings_ms_us_epsg5070_all.parquet for the real GEOID/coords.
SORT_COL = {"label": "Rel_Score", "pred": "predicted_value",
            "residual": "residual", "year": "year"}

def resolve_city(city):
    """cbsa code (int) or title substring -> cbsa code; raises with menu on miss."""
    if city is None:
        raise ValueError("Set CITY in the config cell (code or title substring).")
    if isinstance(city, str):
        hits = sorted(set(df.loc[df["city"].str.contains(city, case=False), "cbsa_code"]))
        if len(hits) > 1:  # substring collision (e.g. Dayton / DaytonA Beach): prefer prefix
            starts = sorted(set(df.loc[df["city"].str.lower().str.startswith(city.lower()),
                                       "cbsa_code"]))
            if len(starts) == 1:
                hits = starts
        if len(hits) != 1:
            names = [TITLE.get(int(h), h) for h in hits]
            raise ValueError(f"CITY={city!r} matches {names}; be more specific.")
        return int(hits[0])
    return int(city)

def city_pages(cbsa, sort_by=SORT_BY, page_size=PAGE_SIZE):
    """Rows of one city (both val sets), sorted, chunked into pages."""
    sub = df[df["cbsa_code"].astype(int) == int(cbsa)].sort_values(
        SORT_COL[sort_by], na_position="last").reset_index(drop=True)
    return [sub.iloc[i:i + page_size] for i in range(0, len(sub), page_size)]

def to_rgb(img_u8):
    """uint8 (C,H,W) tile -> (H,W,3) for imshow (first 3 bands = RGB)."""
    return np.transpose(np.asarray(img_u8[:3]), (1, 2, 0))

def render_page(page, cbsa, page_no, n_pages, ncols=6):
    nrows = int(np.ceil(len(page) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 3.0 * nrows), squeeze=False)
    rho = SUMMARY.loc[SUMMARY["cbsa_code"].astype(int) == int(cbsa), "rho"].mean()
    fig.suptitle(f"{TITLE.get(int(cbsa), cbsa)}  [{BRACKET.get(int(cbsa), '?')}]  "
                 f"rho={rho:.2f}  — page {page_no + 1}/{n_pages}, sorted by {SORT_BY}",
                 fontsize=13)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), page.iterrows()):
        ax.imshow(to_rgb(IMAGES[int(r["img_idx"])]))
        pred = "—" if pd.isna(r["predicted_value"]) else f"{r['predicted_value']:+.2f}"
        res = "" if pd.isna(r["residual"]) else f"  r={r['residual']:+.2f}"
        ax.set_title(f"z={r['Rel_Score']:+.2f}  ŷ={pred}{res}\n"
                     f"{int(r['year'])}  bld {int(r['building_id'])}", fontsize=7)
        if pd.notna(r["residual"]) and abs(r["residual"]) > 1.0:
            for s in ax.spines.values():
                s.set(visible=True, color="#c0304a", linewidth=2.5)
            ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

In [ ]:
# ── Render CITY in pages (PAGES=None -> every page) ──────────────────────────────
if CITY is None:
    print("CITY is None — pick one from the summary table above, e.g. CITY = 'Dayton'.")
else:
    _cbsa = resolve_city(CITY)
    _pages = city_pages(_cbsa)
    _idx = range(len(_pages)) if PAGES is None else PAGES
    for _p in _idx:
        render_page(_pages[_p], _cbsa, _p, len(_pages))

In [ ]:
# ── Optional: interactive browser (pip install ipywidgets) ─────────────────────────
try:
    import ipywidgets as widgets

    _options = [(f"{TITLE.get(int(c), c)} (rho={r:.2f}, bld={int(b)})", int(c))
                for c, r, b in SUMMARY.groupby("cbsa_code")
                                      .agg(rho=("rho", "mean"), b=("n_bld", "sum"))
                                      .reset_index().itertuples(index=False)]

    def _browse(cbsa, sort_by, page):
        pages = city_pages(cbsa, sort_by=sort_by)
        render_page(pages[min(page, len(pages) - 1)], cbsa,
                    min(page, len(pages) - 1), len(pages))

    widgets.interact(_browse,
                     cbsa=widgets.Dropdown(options=_options, description="city"),
                     sort_by=widgets.Dropdown(options=list(SORT_COL), value=SORT_BY,
                                              description="sort"),
                     page=widgets.IntSlider(0, 0, 60, 1, description="page"))
except ImportError:
    print("ipywidgets not installed — use the paged cell above, or "
          "`pip install ipywidgets` and re-run this cell for the slider UI.")